# Identity Trust Assessment Layer — Prototype

*Author:* Hamza Shekhani  
*Version:* 0.1  
*Date:* 2026-09-11  

*Purpose:* Reference implementation of a five-dimension trust-assessment 
layer for patient identity resolution in the Malaffi Health Information 
Exchange, per the DOH Standard on Patient Healthcare Data Privacy 
(DOH/SD/SS/PHDP/0.9).

*Sections:*
1. Configuration
2. Data structures
3. Trust validators
4. Composite trust score
5. Evaluation on synthetic population
6. Results and figures

In [13]:
# ============================================================
# Cell 2 — Configuration
# ============================================================
# Sets up the environment. Fixed random seed ensures every run
# produces identical results, which is required for reproducibility.

import random
import uuid
from dataclasses import dataclass, field
from typing import Optional

# Reproducibility — DO NOT CHANGE THIS VALUE
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# DOH Standard version this prototype implements
DOH_STANDARD_VERSION = "DOH/SD/SS/PHDP/0.9"

print("Prototype initialised.")
print(f"DOH Standard: {DOH_STANDARD_VERSION}")
print(f"Random seed: {RANDOM_SEED}")

Prototype initialised.
DOH Standard: DOH/SD/SS/PHDP/0.9
Random seed: 42


In [14]:
# ============================================================
# Cell 3 — PatientRecord data structure
# ============================================================
# Defines what a "patient record" looks like in this prototype.
# Every record has identifiers, demographics, and metadata —
# the same fields Malaffi receives from a source system.

@dataclass
class PatientRecord:
    # --- Identifiers ---
    emirates_id: Optional[str] = None
    passport_number: Optional[str] = None
    local_mrn: Optional[str] = None

    # --- Demographics ---
    given_name: str = ""
    family_name: str = ""
    date_of_birth: str = ""        # YYYY-MM-DD
    nationality: str = ""

    # --- Metadata ---
    source_facility: str = ""
    registration_date: str = ""    # YYYY-MM-DD

    # --- Ground truth (for evaluation only) ---
    canonical_id: str = ""         # TRUE identity of this patient


# Quick test — create one record
test_record = PatientRecord(
    emirates_id="784-1985-1234567-1",
    given_name="Ahmed",
    family_name="Al-Mansoori",
    date_of_birth="1985-03-15",
    nationality="UAE",
    source_facility="Cleveland Clinic Abu Dhabi",
    registration_date="2024-01-10",
    canonical_id="PATIENT_001"
)

print("Test record created:")
print(test_record)

Test record created:
PatientRecord(emirates_id='784-1985-1234567-1', passport_number=None, local_mrn=None, given_name='Ahmed', family_name='Al-Mansoori', date_of_birth='1985-03-15', nationality='UAE', source_facility='Cleveland Clinic Abu Dhabi', registration_date='2024-01-10', canonical_id='PATIENT_001')


In [15]:
# ============================================================
# Cell 4 — Completeness Validator
# ============================================================
# Implements DOH Standard clauses:
#   DP-DA-05    — periodic assessment of data accuracy
#   DP-PDC-01   — documenting personal data processing
#
# Checks that a patient record contains the mandatory identity
# fields required for Malaffi ingestion.
#
# Returns a score in [0, 1]:
#   1.0  — Emirates ID present (primary identifier)
#   0.7  — Passport present but no Emirates ID (fallback)
#   0.0  — neither present (unidentifiable)

def validate_completeness(record: PatientRecord) -> float:
    """
    Score a record on completeness of its identity fields.
    See DOH clauses DP-DA-05 and DP-PDC-01.
    """
    # Primary identifier check
    has_emirates_id = record.emirates_id is not None and record.emirates_id.strip() != ""
    has_passport    = record.passport_number is not None and record.passport_number.strip() != ""

    # Name and DOB are also mandatory
    has_name = (record.given_name.strip() != "" and record.family_name.strip() != "")
    has_dob  = record.date_of_birth.strip() != ""

    # If name or DOB missing, the record cannot be trusted at all
    if not has_name or not has_dob:
        return 0.0

    # Score based on which identifier is available
    if has_emirates_id:
        return 1.0
    elif has_passport:
        return 0.7
    else:
        return 0.0


# ------------------------------------------------------------
# Self-test — build 4 records with known expected scores
# ------------------------------------------------------------

test_cases = [
    PatientRecord(
        emirates_id="784-1985-1234567-1", given_name="Ahmed",
        family_name="Al-Mansoori", date_of_birth="1985-03-15"
    ),  # expect 1.0

    PatientRecord(
        passport_number="AB1234567", given_name="Raj",
        family_name="Kumar", date_of_birth="1990-07-22"
    ),  # expect 0.7

    PatientRecord(
        given_name="Maria", family_name="Santos",
        date_of_birth="1992-11-03"
    ),  # expect 0.0  (no identifier)

    PatientRecord(
        emirates_id="784-1985-1234567-1", given_name="",
        family_name="Al-Mansoori", date_of_birth="1985-03-15"
    ),  # expect 0.0  (no given name)
]

expected = [1.0, 0.7, 0.0, 0.0]

print("Completeness validator — self test")
print("-" * 50)

for i, record in enumerate(test_cases):
    score = validate_completeness(record)
    status = "PASS" if score == expected[i] else "FAIL"
    print(f"Case {i+1}: score={score:.1f}  expected={expected[i]:.1f}  [{status}]")

# Assertion — this will raise an error if any case fails
for i, record in enumerate(test_cases):
    assert validate_completeness(record) == expected[i], f"Case {i+1} failed"

print("-" * 50)
print("All tests passed.")

Completeness validator — self test
--------------------------------------------------
Case 1: score=1.0  expected=1.0  [PASS]
Case 2: score=0.7  expected=0.7  [PASS]
Case 3: score=0.0  expected=0.0  [PASS]
Case 4: score=0.0  expected=0.0  [PASS]
--------------------------------------------------
All tests passed.


In [16]:
# ============================================================
# Cell 5 — Temporal Validity Validator
# ============================================================
# Implements DOH Standard clauses:
#   §1.4.2  — Integrity (data not modified, altered, or damaged)
#   DP-DA-04 — patient's right to determine data correctness
#
# Checks that dates within a record make logical sense.
# Returns a score in [0, 1].

from datetime import datetime

def validate_temporal(record: PatientRecord) -> float:
    """
    Score a record on temporal coherence.
    See DOH §1.4.2 (Integrity) and DP-DA-04 (correctness).
    """
    score = 1.0

    # --- Check 1: DOB must be a valid date ---
    try:
        dob = datetime.strptime(record.date_of_birth, "%Y-%m-%d")
    except (ValueError, TypeError):
        return 0.0   # invalid or missing DOB → cannot assess

    # --- Check 2: DOB must not be in the future ---
    today = datetime.today()
    if dob > today:
        return 0.0

    # --- Check 3: DOB must imply a plausible age (0–130 years) ---
    age_years = (today - dob).days / 365.25
    if age_years < 0 or age_years > 130:
        return 0.0

    # --- Check 4: Registration date must exist and be valid ---
    try:
        reg = datetime.strptime(record.registration_date, "%Y-%m-%d")
    except (ValueError, TypeError):
        score -= 0.3   # missing registration date is a partial penalty

    # --- Check 5: Registration cannot precede birth ---
    if record.registration_date:
        try:
            reg = datetime.strptime(record.registration_date, "%Y-%m-%d")
            if reg < dob:
                score -= 0.5   # registration before birth is a serious error
        except (ValueError, TypeError):
            pass

    # Clamp to [0, 1]
    return max(0.0, min(1.0, score))


# ------------------------------------------------------------
# Self-test
# ------------------------------------------------------------

temporal_tests = [
    PatientRecord(
        given_name="Ahmed", family_name="Al-Mansoori",
        date_of_birth="1985-03-15", registration_date="2024-01-10"
    ),  # expect 1.0

    PatientRecord(
        given_name="Raj", family_name="Kumar",
        date_of_birth="1990-07-22", registration_date=""
    ),  # expect 0.7  (missing registration date)

    PatientRecord(
        given_name="Maria", family_name="Santos",
        date_of_birth="2030-01-01", registration_date="2024-01-10"
    ),  # expect 0.0  (DOB in future)

    PatientRecord(
        given_name="Li", family_name="Wei",
        date_of_birth="1995-06-15", registration_date="1990-01-01"
    ),  # expect 0.5  (registration before birth)

    PatientRecord(
        given_name="Fatima", family_name="Al-Zahra",
        date_of_birth="not-a-date", registration_date="2024-01-10"
    ),  # expect 0.0  (invalid DOB format)
]

temporal_expected = [1.0, 0.7, 0.0, 0.5, 0.0]

print("Temporal validity validator — self test")
print("-" * 50)

for i, record in enumerate(temporal_tests):
    score = validate_temporal(record)
    status = "PASS" if abs(score - temporal_expected[i]) < 0.001 else "FAIL"
    print(f"Case {i+1}: score={score:.2f}  expected={temporal_expected[i]:.2f}  [{status}]")

for i, record in enumerate(temporal_tests):
    assert abs(validate_temporal(record) - temporal_expected[i]) < 0.001, f"Case {i+1} failed"

print("-" * 50)
print("All tests passed.")

Temporal validity validator — self test
--------------------------------------------------
Case 1: score=1.00  expected=1.00  [PASS]
Case 2: score=0.70  expected=0.70  [PASS]
Case 3: score=0.00  expected=0.00  [PASS]
Case 4: score=0.50  expected=0.50  [PASS]
Case 5: score=0.00  expected=0.00  [PASS]
--------------------------------------------------
All tests passed.


In [17]:
# ============================================================
# Cell 6 — Identity Consistency Validator
# ============================================================
# Implements DOH Standard clauses:
#   DP-DA-02 — validate identity before responding to data access
#   §4.2.3.2 — patient identity must not be disclosed improperly
#
# Checks that identifiers and demographics agree with each other
# within the same record. Returns a score in [0, 1].

import re

def validate_identity_consistency(record: PatientRecord) -> float:
    """
    Score a record on internal consistency between identifiers
    and demographics. See DOH DP-DA-02 and §4.2.3.2.
    """
    score = 1.0
    checks_performed = 0

    # --- Check 1: Emirates ID format ---
    # UAE Emirates ID format: 784-YYYY-NNNNNNN-C  (15 digits total)
    if record.emirates_id:
        checks_performed += 1
        emirates_pattern = r"^784-\d{4}-\d{7}-\d$"
        if not re.match(emirates_pattern, record.emirates_id):
            score -= 0.5   # malformed Emirates ID

    # --- Check 2: Passport format (generic — 6-9 alphanumeric chars) ---
    if record.passport_number:
        checks_performed += 1
        passport_pattern = r"^[A-Z0-9]{6,9}$"
        if not re.match(passport_pattern, record.passport_number):
            score -= 0.3

    # --- Check 3: Name fields must not be identical strings ---
    # (Given name = family name is almost always a data entry error)
    if record.given_name and record.family_name:
        checks_performed += 1
        if record.given_name.strip().lower() == record.family_name.strip().lower():
            score -= 0.4

    # --- Check 4: Nationality must be a plausible ISO-style code ---
    # Accepts 2-3 letter codes (AE, UAE, IN, PK, etc.) or full country names
    if record.nationality:
        checks_performed += 1
        nat = record.nationality.strip()
        if len(nat) < 2:
            score -= 0.3

    # --- If no checks were performed, defer to other dimensions ---
    if checks_performed == 0:
        return 0.5

    return max(0.0, min(1.0, score))


# ------------------------------------------------------------
# Self-test
# ------------------------------------------------------------

identity_tests = [
    PatientRecord(
        emirates_id="784-1985-1234567-1",
        given_name="Ahmed", family_name="Al-Mansoori",
        nationality="UAE"
    ),  # expect 1.0

    PatientRecord(
        emirates_id="123-4567-890",
        given_name="Raj", family_name="Kumar",
        nationality="IN"
    ),  # expect 0.5  (malformed Emirates ID)

    PatientRecord(
        passport_number="AB1234567",
        given_name="Maria", family_name="Santos",
        nationality="PH"
    ),  # expect 1.0

    PatientRecord(
        passport_number="ab-12",
        given_name="Li", family_name="Wei",
        nationality="CN"
    ),  # expect 0.7  (malformed passport)

    PatientRecord(
        emirates_id="784-1985-1234567-1",
        given_name="Smith", family_name="Smith",
        nationality="UAE"
    ),  # expect 0.6  (given = family name)

    PatientRecord(
        given_name="Ahmed", family_name="Al-Mansoori",
        nationality=""
    ),  # expect 1.0  (no identifiers to check, but name present)
]

# Manually computed expected scores
identity_expected = [1.0, 0.5, 1.0, 0.7, 0.6, 1.0]

print("Identity consistency validator — self test")
print("-" * 50)

for i, record in enumerate(identity_tests):
    score = validate_identity_consistency(record)
    status = "PASS" if abs(score - identity_expected[i]) < 0.001 else "FAIL"
    print(f"Case {i+1}: score={score:.2f}  expected={identity_expected[i]:.2f}  [{status}]")

for i, record in enumerate(identity_tests):
    assert abs(validate_identity_consistency(record) - identity_expected[i]) < 0.001, f"Case {i+1} failed"

print("-" * 50)
print("All tests passed.")

Identity consistency validator — self test
--------------------------------------------------
Case 1: score=1.00  expected=1.00  [PASS]
Case 2: score=0.50  expected=0.50  [PASS]
Case 3: score=1.00  expected=1.00  [PASS]
Case 4: score=0.70  expected=0.70  [PASS]
Case 5: score=0.60  expected=0.60  [PASS]
Case 6: score=1.00  expected=1.00  [PASS]
--------------------------------------------------
All tests passed.


In [18]:
# ============================================================
# Cell 7 — Provenance Validator
# ============================================================
# Implements DOH Standard clauses:
#   DP-IS-02  — document owners/operators of systems handling data
#   DP-IS-03  — evaluate role in the data processing ecosystem
#   DP-DA-06  — third-party handling with documented agreements
#
# Checks that the record's source facility is a known, registered
# provider with an established data quality tier.
# Returns a score in [0, 1].

# ------------------------------------------------------------
# Simulated DOH provider registry
# ------------------------------------------------------------
# In production, this would be queried from the DOH registry.
# Here it is hard-coded to simulate three maturity tiers.

DOH_PROVIDER_REGISTRY = {
    # High-maturity facilities — mature EMR, long-standing connectivity
    "Cleveland Clinic Abu Dhabi": "high",
    "Sheikh Khalifa Medical City": "high",
    "SSMC": "high",

    # Medium-maturity — connected, but known to have occasional data issues
    "Al Noor Hospital": "medium",
    "Mediclinic Abu Dhabi": "medium",
    "NMC Royal Hospital": "medium",

    # Low-maturity — recent additions, minimal integration history
    "Community Clinic Al Ain": "low",
    "Rural Health Centre Liwa": "low",
}

# Score assigned to each tier
TIER_SCORES = {
    "high":   1.0,
    "medium": 0.7,
    "low":    0.4,
}

# Score for a facility that is listed but not in the registry
# (e.g., new provider, registry lag — not necessarily fraudulent)
UNREGISTERED_SOURCE_SCORE = 0.3


def validate_provenance(record: PatientRecord) -> float:
    """
    Score a record on source provenance.
    See DOH DP-IS-02, DP-IS-03, DP-DA-06.
    """
    # --- Check 1: source facility must be present ---
    if not record.source_facility or record.source_facility.strip() == "":
        return 0.0   # missing — data quality failure

    # --- Check 2: source facility must be in the registry ---
    facility = record.source_facility.strip()
    if facility not in DOH_PROVIDER_REGISTRY:
        return UNREGISTERED_SOURCE_SCORE   # 0.3 — possibly registry lag

    # --- Check 3: return tier-based score ---
    tier = DOH_PROVIDER_REGISTRY[facility]
    return TIER_SCORES[tier]


# ------------------------------------------------------------
# Self-test
# ------------------------------------------------------------

provenance_tests = [
    PatientRecord(source_facility="Cleveland Clinic Abu Dhabi"),   # expect 1.0
    PatientRecord(source_facility="Mediclinic Abu Dhabi"),         # expect 0.7
    PatientRecord(source_facility="Rural Health Centre Liwa"),     # expect 0.4
    PatientRecord(source_facility="Some Unknown Clinic"),          # expect 0.3
    PatientRecord(source_facility=""),                             # expect 0.0
]

provenance_expected = [1.0, 0.7, 0.4, 0.3, 0.0]

print("Provenance validator — self test")
print("-" * 50)

for i, record in enumerate(provenance_tests):
    score = validate_provenance(record)
    status = "PASS" if abs(score - provenance_expected[i]) < 0.001 else "FAIL"
    facility = record.source_facility if record.source_facility else "(none)"
    print(f"Case {i+1}: {facility:<32} score={score:.2f}  [{status}]")

for i, record in enumerate(provenance_tests):
    assert abs(validate_provenance(record) - provenance_expected[i]) < 0.001, f"Case {i+1} failed"

print("-" * 50)
print("All tests passed.")


Provenance validator — self test
--------------------------------------------------
Case 1: Cleveland Clinic Abu Dhabi       score=1.00  [PASS]
Case 2: Mediclinic Abu Dhabi             score=0.70  [PASS]
Case 3: Rural Health Centre Liwa         score=0.40  [PASS]
Case 4: Some Unknown Clinic              score=0.30  [PASS]
Case 5: (none)                           score=0.00  [PASS]
--------------------------------------------------
All tests passed.


In [19]:
class CrossRecordValidator:

    def __init__(self):
        self.emirates_id_index = {}
        self.name_dob_index = {}

    def add_record(self, record):
        if record.emirates_id:
            self.emirates_id_index[record.emirates_id] = record.canonical_id

        key = (
            record.given_name.strip().lower(),
            record.family_name.strip().lower(),
            record.date_of_birth
        )
        self.name_dob_index[key] = record.canonical_id

    def validate(self, record):
        if record.emirates_id and record.emirates_id in self.emirates_id_index:
            existing = self.emirates_id_index[record.emirates_id]
            if existing != record.canonical_id:
                return 0.0

        key = (
            record.given_name.strip().lower(),
            record.family_name.strip().lower(),
            record.date_of_birth
        )
        if key in self.name_dob_index:
            existing = self.name_dob_index[key]
            if existing != record.canonical_id:
                return 0.5

        return 1.0


validator = CrossRecordValidator()

patient_a = PatientRecord(
    emirates_id="784-1985-1234567-1",
    given_name="Ahmed", family_name="Al-Mansoori",
    date_of_birth="1985-03-15",
    canonical_id="PATIENT_A"
)
patient_b = PatientRecord(
    emirates_id="784-1990-7654321-2",
    given_name="Raj", family_name="Kumar",
    date_of_birth="1990-07-22",
    canonical_id="PATIENT_B"
)

validator.add_record(patient_a)
validator.add_record(patient_b)

tests = [
    PatientRecord(
        emirates_id="784-1985-1234567-1",
        given_name="Ahmed", family_name="Al-Mansoori",
        date_of_birth="1985-03-15",
        canonical_id="PATIENT_A"
    ),
    PatientRecord(
        emirates_id="784-1985-1234567-1",
        given_name="Fatima", family_name="Al-Zahra",
        date_of_birth="1992-01-01",
        canonical_id="PATIENT_C"
    ),
    PatientRecord(
        emirates_id="784-1999-9999999-9",
        given_name="Ahmed", family_name="Al-Mansoori",
        date_of_birth="1985-03-15",
        canonical_id="PATIENT_D"
    ),
    PatientRecord(
        emirates_id="784-2000-1111111-1",
        given_name="Li", family_name="Wei",
        date_of_birth="2000-12-05",
        canonical_id="PATIENT_E"
    ),
]

expected = [1.0, 0.0, 0.5, 1.0]

print("Cross-record consistency validator - self test")
print("-" * 50)

for i, record in enumerate(tests):
    score = validator.validate(record)
    status = "PASS" if abs(score - expected[i]) < 0.001 else "FAIL"
    print(f"Case {i+1}: score={score:.2f}  expected={expected[i]:.2f}  [{status}]")

for i, record in enumerate(tests):
    assert abs(validator.validate(record) - expected[i]) < 0.001, f"Case {i+1} failed"

print("-" * 50)
print("All tests passed.")

Cross-record consistency validator - self test
--------------------------------------------------
Case 1: score=1.00  expected=1.00  [PASS]
Case 2: score=0.00  expected=0.00  [PASS]
Case 3: score=0.50  expected=0.50  [PASS]
Case 4: score=1.00  expected=1.00  [PASS]
--------------------------------------------------
All tests passed.


In [20]:
# ============================================================
# Cell 9 - Composite Trust Score
# ============================================================
# Combines the five validator outputs into a single weighted score.
# See Section 3.4.6 of the methodology.

DEFAULT_WEIGHTS = {
    "completeness": 0.20,
    "temporal":     0.15,
    "identity":     0.20,
    "provenance":   0.15,
    "cross_record": 0.30,
}

# Sanity checks
assert abs(sum(DEFAULT_WEIGHTS.values()) - 1.0) < 0.001, "Weights must sum to 1.0"
assert max(DEFAULT_WEIGHTS.values()) <= 0.40, "No weight may exceed 0.40"


def compute_trust_score(completeness, temporal, identity, provenance, cross_record):
    score = (
        DEFAULT_WEIGHTS["completeness"] * completeness +
        DEFAULT_WEIGHTS["temporal"]     * temporal +
        DEFAULT_WEIGHTS["identity"]     * identity +
        DEFAULT_WEIGHTS["provenance"]   * provenance +
        DEFAULT_WEIGHTS["cross_record"] * cross_record
    )
    return round(score, 4)


# ------------------------------------------------------------
# Self-test
# ------------------------------------------------------------

test_cases = [
    (1.0, 1.0, 1.0, 1.0, 1.0, 1.0),
    (0.0, 0.0, 0.0, 0.0, 0.0, 0.0),
    (1.0, 1.0, 1.0, 1.0, 0.0, 0.70),
    (0.0, 1.0, 1.0, 1.0, 1.0, 0.80),
]

print("Composite trust score - self test")
print("-" * 50)

for i, (c, t, idn, p, cr, expected) in enumerate(test_cases):
    score = compute_trust_score(c, t, idn, p, cr)
    status = "PASS" if abs(score - expected) < 0.001 else "FAIL"
    print(f"Case {i+1}: score={score:.4f}  expected={expected:.4f}  [{status}]")

for i, (c, t, idn, p, cr, expected) in enumerate(test_cases):
    assert abs(compute_trust_score(c, t, idn, p, cr) - expected) < 0.001, f"Case {i+1} failed"

print("-" * 50)
print("All tests passed.")

Composite trust score - self test
--------------------------------------------------
Case 1: score=1.0000  expected=1.0000  [PASS]
Case 2: score=0.0000  expected=0.0000  [PASS]
Case 3: score=0.7000  expected=0.7000  [PASS]
Case 4: score=0.8000  expected=0.8000  [PASS]
--------------------------------------------------
All tests passed.


In [21]:
# ============================================================
# Cell 10 - Decision Router
# ============================================================
# Maps the composite trust score to one of three outcomes.
# See Section 3.4.7 of the methodology.

LOW_RISK_THRESHOLD    = 0.85
MEDIUM_RISK_THRESHOLD = 0.50


def route_decision(trust_score):
    if trust_score >= LOW_RISK_THRESHOLD:
        return "AUTO_LINK"
    elif trust_score >= MEDIUM_RISK_THRESHOLD:
        return "LINK_WITH_FLAG"
    else:
        return "QUARANTINE"


# ------------------------------------------------------------
# Self-test
# ------------------------------------------------------------

router_tests = [
    (0.95, "AUTO_LINK"),
    (0.85, "AUTO_LINK"),
    (0.84, "LINK_WITH_FLAG"),
    (0.50, "LINK_WITH_FLAG"),
    (0.49, "QUARANTINE"),
    (0.10, "QUARANTINE"),
    (0.00, "QUARANTINE"),
]

print("Decision router - self test")
print("-" * 50)

for i, (score, expected) in enumerate(router_tests):
    decision = route_decision(score)
    status = "PASS" if decision == expected else "FAIL"
    print(f"Case {i+1}: score={score:.2f}  decision={decision:<16} [{status}]")

for i, (score, expected) in enumerate(router_tests):
    assert route_decision(score) == expected, f"Case {i+1} failed"

print("-" * 50)
print("All tests passed.")

Decision router - self test
--------------------------------------------------
Case 1: score=0.95  decision=AUTO_LINK        [PASS]
Case 2: score=0.85  decision=AUTO_LINK        [PASS]
Case 3: score=0.84  decision=LINK_WITH_FLAG   [PASS]
Case 4: score=0.50  decision=LINK_WITH_FLAG   [PASS]
Case 5: score=0.49  decision=QUARANTINE       [PASS]
Case 6: score=0.10  decision=QUARANTINE       [PASS]
Case 7: score=0.00  decision=QUARANTINE       [PASS]
--------------------------------------------------
All tests passed.


In [22]:
# ============================================================
# Cell 11 - Synthetic Population Generator
# ============================================================
# Generates a synthetic patient population calibrated to UAE
# demographic distributions. Each patient appears across four
# simulated source facilities with realistic degradation.

# ------------------------------------------------------------
# Name pools by demographic group
# ------------------------------------------------------------

NAME_POOLS = {
    "Indian": {
        "given": ["Raj", "Priya", "Arun", "Deepa", "Vikram", "Meera", "Sanjay", "Kavita"],
        "family": ["Kumar", "Sharma", "Patel", "Reddy", "Nair", "Singh", "Iyer", "Gupta"],
    },
    "Pakistani": {
        "given": ["Ali", "Fatima", "Hassan", "Zara", "Bilal", "Aisha", "Imran", "Sana"],
        "family": ["Khan", "Ahmed", "Malik", "Hussain", "Sheikh", "Raza"],
    },
    "Emirati": {
        "given": ["Ahmed", "Maryam", "Khalid", "Noura", "Saeed", "Latifa", "Hamdan", "Shamma"],
        "family": ["Al-Mansoori", "Al-Zahra", "Al-Nuaimi", "Al-Falasi", "Al-Suwaidi", "Al-Mazrouei"],
    },
    "Filipino": {
        "given": ["Maria", "Jose", "Ana", "Carlo", "Rosa", "Miguel", "Liza", "Ramon"],
        "family": ["Santos", "Reyes", "Cruz", "Garcia", "Mendoza", "Torres"],
    },
    "Bangladeshi": {
        "given": ["Rahman", "Nasrin", "Karim", "Salma", "Hasan", "Rima"],
        "family": ["Islam", "Ahmed", "Hossain", "Akter", "Rahman"],
    },
    "Egyptian": {
        "given": ["Mohamed", "Layla", "Omar", "Nour", "Youssef", "Dina"],
        "family": ["Hassan", "Ibrahim", "Mahmoud", "Abdel-Rahman", "Farouk"],
    },
    "Chinese": {
        "given": ["Wei", "Mei", "Chen", "Xiu", "Jun", "Ying"],
        "family": ["Zhang", "Wang", "Li", "Chen", "Liu", "Huang"],
    },
    "Western": {
        "given": ["John", "Sarah", "Michael", "Emma", "David", "Anna"],
        "family": ["Smith", "Johnson", "Williams", "Brown", "Jones"],
    },
}

DEMOGRAPHIC_MIX = [
    ("Indian",     0.38),
    ("Pakistani",  0.17),
    ("Emirati",    0.11),
    ("Bangladeshi", 0.07),
    ("Filipino",   0.06),
    ("Egyptian",   0.05),
    ("Chinese",    0.03),
    ("Western",    0.13),
]

SOURCE_FACILITIES = [
    "Cleveland Clinic Abu Dhabi",
    "Sheikh Khalifa Medical City",
    "SSMC",
    "Al Noor Hospital",
    "Mediclinic Abu Dhabi",
    "NMC Royal Hospital",
    "Community Clinic Al Ain",
    "Rural Health Centre Liwa",
]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def pick_demographic():
    r = random.random()
    cumulative = 0.0
    for group, weight in DEMOGRAPHIC_MIX:
        cumulative += weight
        if r <= cumulative:
            return group
    return "Western"


def make_emirates_id():
    # Format: 784-YYYY-NNNNNNN-C  (15 digits)
    year = random.randint(1950, 2010)
    seven = random.randint(1000000, 9999999)
    check = random.randint(0, 9)
    return f"784-{year}-{seven}-{check}"


def make_dob():
    year = random.randint(1950, 2010)
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    return f"{year}-{month:02d}-{day:02d}"


def pick_facility():
    return random.choice(SOURCE_FACILITIES)


# ------------------------------------------------------------
# Canonical patient generation (ground truth)
# ------------------------------------------------------------

def generate_canonical_patients(n):
    patients = []
    for i in range(n):
        group = pick_demographic()
        pool = NAME_POOLS[group]
        patients.append({
            "canonical_id": f"PATIENT_{i:04d}",
            "demographic": group,
            "given_name": random.choice(pool["given"]),
            "family_name": random.choice(pool["family"]),
            "date_of_birth": make_dob(),
            "nationality": group,
            "emirates_id": make_emirates_id(),
        })
    return patients


# ------------------------------------------------------------
# Source record generation (with realistic degradation)
# ------------------------------------------------------------

def degrade_record(patient, source_index):
    # Start from the clean canonical values
    record = PatientRecord(
        emirates_id=patient["emirates_id"],
        given_name=patient["given_name"],
        family_name=patient["family_name"],
        date_of_birth=patient["date_of_birth"],
        nationality=patient["nationality"],
        source_facility=pick_facility(),
        registration_date="2024-01-10",
        canonical_id=patient["canonical_id"],
    )

    # Degradation 1: Emirates ID missing (25% of records)
    if random.random() < 0.25:
        record.emirates_id = None

    # Degradation 2: Emirates ID malformed (5%)
    elif random.random() < 0.05:
        record.emirates_id = "INVALID_784_ABC"

    # Degradation 3: DOB shifted (10%)
    if random.random() < 0.10:
        # shift by 1-5 days
        parts = record.date_of_birth.split("-")
        day = int(parts[2])
        day = max(1, day - random.randint(1, 5))
        record.date_of_birth = f"{parts[0]}-{parts[1]}-{day:02d}"

    # Degradation 4: Given name variant (15%)
    if random.random() < 0.15:
        record.given_name = record.given_name[:-1] + "x"

    return record


# ------------------------------------------------------------
# Run it
# ------------------------------------------------------------

random.seed(RANDOM_SEED)

N_PATIENTS = 100
SOURCES_PER_PATIENT = 4

canonical_patients = generate_canonical_patients(N_PATIENTS)

all_source_records = []
for patient in canonical_patients:
    for s in range(SOURCES_PER_PATIENT):
        rec = degrade_record(patient, s)
        all_source_records.append(rec)

print("Synthetic population generated")
print("-" * 50)
print(f"Canonical patients: {len(canonical_patients)}")
print(f"Source records:     {len(all_source_records)}")
print(f"Records per patient: {SOURCES_PER_PATIENT}")
print("-" * 50)
print()
print("Sample record (record 0):")
print(all_source_records[0])
print()
print("Degradation summary:")
missing_eid = sum(1 for r in all_source_records if not r.emirates_id)
print(f"  Records missing Emirates ID: {missing_eid} ({100*missing_eid/len(all_source_records):.1f}%)")

Synthetic population generated
--------------------------------------------------
Canonical patients: 100
Source records:     400
Records per patient: 4
--------------------------------------------------

Sample record (record 0):
PatientRecord(emirates_id=None, passport_number=None, local_mrn=None, given_name='Ahmed', family_name='Al-Mazrouei', date_of_birth='1967-04-08', nationality='Emirati', source_facility='Cleveland Clinic Abu Dhabi', registration_date='2024-01-10', canonical_id='PATIENT_0000')

Degradation summary:
  Records missing Emirates ID: 89 (22.2%)


In [23]:
# ============================================================
# Cell 11b - Collision Injection
# ============================================================
# Injects realistic identity collisions into the population.
# Real-world analogue: family members sharing an insurance card,
# transcription errors that copy one patient's ID into another's.

COLLISION_RATE = 0.10

collision_pairs = []

for i, patient in enumerate(canonical_patients):
    if random.random() < COLLISION_RATE:
        donor_idx = random.randint(0, len(canonical_patients) - 1)
        while donor_idx == i:
            donor_idx = random.randint(0, len(canonical_patients) - 1)

        donor = canonical_patients[donor_idx]
        patient["emirates_id"] = donor["emirates_id"]
        patient["family_name"] = donor["family_name"]
        collision_pairs.append((patient["canonical_id"], donor["canonical_id"]))

print(f"Collision injections: {len(collision_pairs)}")
for a, b in collision_pairs[:5]:
    print(f"  {a} shares Emirates ID with {b}")

# Regenerate source records with the modified canonicals
all_source_records = []
for patient in canonical_patients:
    for s in range(SOURCES_PER_PATIENT):
        rec = degrade_record(patient, s)
        all_source_records.append(rec)

missing_eid = sum(1 for r in all_source_records if not r.emirates_id)
print("Missing Emirates ID:", missing_eid)
print("Total records:", len(all_source_records))

Collision injections: 12
  PATIENT_0016 shares Emirates ID with PATIENT_0008
  PATIENT_0017 shares Emirates ID with PATIENT_0027
  PATIENT_0018 shares Emirates ID with PATIENT_0007
  PATIENT_0029 shares Emirates ID with PATIENT_0010
  PATIENT_0038 shares Emirates ID with PATIENT_0089
Missing Emirates ID: 106
Total records: 400


In [24]:
def baseline_decision(new_record, index_records):
    for idx in index_records:
        if (new_record.emirates_id and idx.emirates_id and
            new_record.emirates_id == idx.emirates_id and
            new_record.family_name.lower() == idx.family_name.lower()):
            return "AUTO_LINK"
    return "QUARANTINE"


evaluation_pairs = []

for cid, recs in records_by_patient.items():
    index_rec = recs[0]
    for rec in recs[1:]:
        evaluation_pairs.append((rec, index_rec, "MATCH"))

patient_ids = list(records_by_patient.keys())

for i in range(0, 20, 2):
    a = records_by_patient[patient_ids[i]][1]
    b = records_by_patient[patient_ids[i+1]][0]
    evaluation_pairs.append((a, b, "NO_MATCH"))

family_groups = {}
for cid, recs in records_by_patient.items():
    fn = recs[0].family_name.lower()
    family_groups.setdefault(fn, []).append(cid)

type_b_count = 0
for fn, cids in family_groups.items():
    if len(cids) >= 2 and type_b_count < 15:
        a = records_by_patient[cids[0]][1]
        b = records_by_patient[cids[1]][0]
        if a.emirates_id != b.emirates_id:
            evaluation_pairs.append((a, b, "NO_MATCH"))
            type_b_count = type_b_count + 1

type_c_count = 0
for pa_id, pb_id in collision_pairs:
    if pa_id in records_by_patient and pb_id in records_by_patient:
        a = records_by_patient[pa_id][1]
        b = records_by_patient[pb_id][0]
        evaluation_pairs.append((a, b, "NO_MATCH"))
        type_c_count = type_c_count + 1

print("Evaluation pairs:", len(evaluation_pairs))
true_matches = sum(1 for p in evaluation_pairs if p[2] == "MATCH")
print("True matches:", true_matches)
print("Non-matches:", len(evaluation_pairs) - true_matches)
print("Type A (easy): 20")
print("Type B (same name):", type_b_count)
print("Type C (same EID):", type_c_count)
print("")

cv_eval = CrossRecordValidator()
for cid, recs in records_by_patient.items():
    cv_eval.add_record(recs[0])

baseline_results = []
trust_results = []

for new_rec, index_rec, truth in evaluation_pairs:
    b_decision = baseline_decision(new_rec, [index_rec])
    baseline_results.append((b_decision == "AUTO_LINK", truth))

    completeness = validate_completeness(new_rec)
    temporal = validate_temporal(new_rec)
    identity = validate_identity_consistency(new_rec)
    provenance = validate_provenance(new_rec)
    cross_record = cv_eval.validate(new_rec)
    trust = compute_trust_score(completeness, temporal, identity, provenance, cross_record)
    t_decision = route_decision(trust)
    trust_results.append((t_decision == "AUTO_LINK", truth))


def compute_metrics(results):
    tp = sum(1 for linked, truth in results if linked and truth == "MATCH")
    fp = sum(1 for linked, truth in results if linked and truth == "NO_MATCH")
    tn = sum(1 for linked, truth in results if not linked and truth == "NO_MATCH")
    fn = sum(1 for linked, truth in results if not linked and truth == "MATCH")
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "precision": precision, "recall": recall,
            "f1": f1, "fpr": fpr, "fnr": fnr}


baseline_metrics = compute_metrics(baseline_results)
trust_metrics = compute_metrics(trust_results)

print("Results comparison")
print("=" * 60)
print("Metric               Baseline    Trust Layer")
print("-" * 60)
print("True Positives  ", baseline_metrics["TP"], "          ", trust_metrics["TP"])
print("False Positives ", baseline_metrics["FP"], "          ", trust_metrics["FP"])
print("True Negatives  ", baseline_metrics["TN"], "          ", trust_metrics["TN"])
print("False Negatives ", baseline_metrics["FN"], "          ", trust_metrics["FN"])
print("-" * 60)
print("Precision       ", round(baseline_metrics["precision"], 3), "        ", round(trust_metrics["precision"], 3))
print("Recall          ", round(baseline_metrics["recall"], 3), "        ", round(trust_metrics["recall"], 3))
print("F1 Score        ", round(baseline_metrics["f1"], 3), "        ", round(trust_metrics["f1"], 3))
print("=" * 60)

NameError: name 'records_by_patient' is not defined